In [1]:
# filtered out the following:
# 1. Possible that changed method is added to fixed code (so not in buggy code) or removed from buggy code (so not in fixed code) so they are not collected in the first place
# 2. We identify the method by project name + method signature + focal method invoked, they are only identical if all 3 are identical, we then take intersection of fixed and buggy bugs based on this identification.
# 3. We remove the bugs where the focal method is private or default, as it is not trivial for generating unit tests for them.

In [2]:
import json
import javalang.tree as ast
import javalang.parse as parse

In [3]:
fixed_source_file = 'data/prompts/fixed_source_data_draft.jsonl'

In [4]:
buggy_source_file = 'data/prompts/buggy_source_data_draft.jsonl'

In [5]:
def is_method_public_or_protected(java_code):
    # Parse the Java code
    wrapped_java_code = 'class Test {\n' + java_code.replace('\r\n', '\n\n') + '\n}'
    tree = parse.parse(wrapped_java_code)
    # search for constructor
    for path, node in tree.filter(ast.ConstructorDeclaration):
        if node:
            if 'public' in node.modifiers or 'protected' in node.modifiers:
                return True
            return False

    # Search for the method
    for path, node in tree.filter(ast.MethodDeclaration):
        if node:
            if 'public' in node.modifiers or 'protected' in node.modifiers:
                return True
            return False
    raise ValueError(f"Method not found in the {java_code}")

In [6]:
is_method_public_or_protected("""BasicBlock(BasicBlock parent, Node root) {
      this.parent = parent;

      // only named functions may be hoisted.
      this.isHoisted = NodeUtil.isHoistedFunctionDeclaration(root);

      this.isFunction = root.getType() == Token.FUNCTION;

      if (root.getParent() != null) {
        int pType = root.getParent().getType();
        this.isLoop = pType == Token.DO ||
            pType == Token.WHILE ||
            pType == Token.FOR;
      } else {
        this.isLoop = false;
      }
    }""")

False

In [7]:
fixed_bugs = {}
with open(fixed_source_file, 'r') as f:
    for line in f:
        loaded_json = json.loads(line)
        if is_method_public_or_protected(loaded_json["source:source_method_code_format"]):
            bug_name = "_".join(loaded_json["extra:project_name"].split("_")[:2])
            fixed_bugs[bug_name+loaded_json["source:source_method_signature"]+str(int(loaded_json["focal_method_invoked"]))] = loaded_json
        else:
            print(f"skipped json with private or default focal method: {loaded_json['source:source_method_signature']}")

skipped json with private or default focal method: com.google.javascript.jscomp#NodeUtil#functionCallHasSideEffects(com.google.javascript.rhino#Node, com.google.javascript.jscomp#AbstractCompiler)
skipped json with private or default focal method: com.google.javascript.jscomp#BasicBlock#provablyExecutesBefore(com.google.javascript.jscomp#BasicBlock)
skipped json with private or default focal method: com.google.javascript.jscomp#ReferenceCollection#isAssignedOnceInLifetime()
skipped json with private or default focal method: com.google.javascript.jscomp#BasicBlock#BasicBlock(com.google.javascript.jscomp#BasicBlock, com.google.javascript.rhino#Node)
skipped json with private or default focal method: com.google.javascript.jscomp#PeepholeFoldConstants#tryFoldShift(com.google.javascript.rhino#Node, com.google.javascript.rhino#Node, com.google.javascript.rhino#Node)
skipped json with private or default focal method: com.google.javascript.rhino.jstype#FunctionType#resolveInternal(com.google.j

In [8]:
len(fixed_bugs)

1070

In [9]:
buggy_bugs = {}
with open(buggy_source_file, 'r') as f:
    for line in f:
        loaded_json = json.loads(line)
        if is_method_public_or_protected(loaded_json["source:source_method_code_format"]):
            bug_name = "_".join(loaded_json["extra:project_name"].split("_")[:2])
            buggy_bugs[bug_name+loaded_json["source:source_method_signature"]+str(int(loaded_json["focal_method_invoked"]))] = loaded_json
        else:
            print(f"skipped json with private or default focal method: {loaded_json['source:source_method_signature']}")

skipped json with private or default focal method: com.google.javascript.jscomp#NodeUtil#functionCallHasSideEffects(com.google.javascript.rhino#Node, com.google.javascript.jscomp#AbstractCompiler)
skipped json with private or default focal method: com.google.javascript.jscomp#ReferenceCollection#isAssignedOnceInLifetime()
skipped json with private or default focal method: com.google.javascript.jscomp#BasicBlock#provablyExecutesBefore(com.google.javascript.jscomp#BasicBlock)
skipped json with private or default focal method: com.google.javascript.jscomp#BasicBlock#BasicBlock(com.google.javascript.jscomp#BasicBlock, com.google.javascript.rhino#Node)
skipped json with private or default focal method: com.google.javascript.jscomp#PeepholeFoldConstants#tryFoldShift(com.google.javascript.rhino#Node, com.google.javascript.rhino#Node, com.google.javascript.rhino#Node)
skipped json with private or default focal method: com.google.javascript.rhino.jstype#FunctionType#resolveInternal(com.google.j

In [10]:
len(buggy_bugs)

919

In [11]:
list(fixed_bugs.keys())[0]

'Lang_30org.apache.commons.lang3#StringUtils#indexOfAnyBut(org.apache.commons.lang3#CharSequence, org.apache.commons.lang3#char[])1'

In [12]:
fixed_bugs[list(fixed_bugs.keys())[0]]

{'extra:project_name': 'Lang_30_fixed',
 'source:source_method_code_format': 'public static int indexOfAnyBut(CharSequence cs, char[] searchChars) {\n        if (isEmpty(cs) || ArrayUtils.isEmpty(searchChars)) {\n            return INDEX_NOT_FOUND;\n        }\n        int csLen = cs.length();\n        int csLast = csLen - 1;\n        int searchLen = searchChars.length;\n        int searchLast = searchLen - 1;\n        outer:\n        for (int i = 0; i < csLen; i++) {\n            char ch = cs.charAt(i);\n            for (int j = 0; j < searchLen; j++) {\n                if (searchChars[j] == ch) {\n                    if (i < csLast && j < searchLast && Character.isHighSurrogate(ch)) {\n                        if (searchChars[j + 1] == cs.charAt(i + 1)) {\n                            continue outer;\n                        }\n                    } else {\n                        continue outer;\n                    }\n                }\n            }\n            return i;\n        }\

In [13]:
list(buggy_bugs.keys())[0]

'Lang_30org.apache.commons.lang3#StringUtils#indexOfAnyBut(java.lang#String, java.lang#String)1'

In [14]:
buggy_bugs[list(buggy_bugs.keys())[0]]

{'extra:project_name': 'Lang_30_buggy',
 'source:source_method_code_format': 'public static int indexOfAnyBut(String str, String searchChars) {\n        if (isEmpty(str) || isEmpty(searchChars)) {\n            return INDEX_NOT_FOUND;\n        }\n        int strLen = str.length();\n        for (int i = 0; i < strLen; i++) {\n            char ch = str.charAt(i);\n            if (searchChars.indexOf(ch) < 0) {\n                    return i;\n            }\n        }\n        return INDEX_NOT_FOUND;\n    }',
 'source:source_method_name': 'indexOfAnyBut',
 'source:source_method_signature': 'org.apache.commons.lang3#StringUtils#indexOfAnyBut(java.lang#String, java.lang#String)',
 'source:source_method_docstring': '/**\n     * <p>Search a String to find the first index of any\n     * character not in the given set of characters.</p>\n     *\n     * <p>A <code>null</code> String will return <code>-1</code>.\n     * A <code>null</code> search string will return <code>-1</code>.</p>\n     *\n   

In [15]:
# get intersection of fixed and buggy bugs
fixed_bugs_keys = set(fixed_bugs.keys())
buggy_bugs_keys = set(buggy_bugs.keys())
intersection = fixed_bugs_keys.intersection(buggy_bugs_keys)
len(intersection)

899

In [16]:
unified_fixed_bugs = []
for key in intersection:
    unified_fixed_bugs.append(fixed_bugs[key])

In [17]:
len(unified_fixed_bugs)

899

In [18]:
unified_buggy_bugs = []
for key in intersection:
    unified_buggy_bugs.append(buggy_bugs[key])

In [19]:
len(unified_buggy_bugs)

899

In [20]:
unified_fixed_bugs[0]

{'extra:project_name': 'Closure_101_fixed',
 'source:source_method_code_format': '@Override\n  protected CompilerOptions createOptions() {\n    CompilerOptions options = new CompilerOptions();\n    options.setCodingConvention(new ClosureCodingConvention());\n    CompilationLevel level = flags.compilation_level;\n    level.setOptionsForCompilationLevel(options);\n    if (flags.debug) {\n      level.setDebugOptionsForCompilationLevel(options);\n    }\n\n    WarningLevel wLevel = flags.warning_level;\n    wLevel.setOptionsForWarningLevel(options);\n    for (FormattingOption formattingOption : flags.formatting) {\n      formattingOption.applyToOptions(options);\n    }\n\n    options.closurePass = flags.process_closure_primitives;\n    initOptionsFromFlags(options);\n    return options;\n  }',
 'source:source_method_name': 'createOptions',
 'source:source_method_signature': 'com.google.javascript.jscomp#CommandLineRunner#createOptions()',
 'source:source_method_docstring': '// expects.',
 '

In [21]:
# sort by extra:project_name
unified_fixed_bugs = sorted(unified_fixed_bugs, key=lambda x: x["extra:project_name"])

In [22]:
fixed_source_file_unified = 'data/prompts/fixed_source_data_unified.jsonl'
with open(fixed_source_file_unified, 'w') as f:
    for bug in unified_fixed_bugs:
        f.write(json.dumps(bug) + '\n')

In [23]:
unified_fixed_invoked = [invoked for invoked in unified_fixed_bugs if invoked["focal_method_invoked"]]

In [24]:
fixed_source_file_unified_invoked = 'data/prompts/fixed_source_data_unified_invoked.jsonl'
with open(fixed_source_file_unified_invoked, 'w') as f:
    for bug in unified_fixed_invoked:
        f.write(json.dumps(bug) + '\n')

In [25]:
# sort by extra:project_name
unified_buggy_bugs = sorted(unified_buggy_bugs, key=lambda x: x["extra:project_name"])

In [26]:
unified_buggy_bugs[0]

{'extra:project_name': 'Chart_10_buggy',
 'source:source_method_code_format': 'public String generateToolTipFragment(String toolTipText) {\n        return " title=\\"" + toolTipText\n            + "\\" alt=\\"\\"";\n    }',
 'source:source_method_name': 'generateToolTipFragment',
 'source:source_method_signature': 'org.jfree.chart.imagemap#StandardToolTipTagFragmentGenerator#generateToolTipFragment(java.lang#String)',
 'source:source_method_docstring': '/**\n     * Generates a tooltip string to go in an HTML image map.\n     *\n     * @param toolTipText  the tooltip.\n     * \n     * @return The formatted HTML area tag attribute(s).\n     */',
 'source:source_method_parameters': [['String', 'toolTipText']],
 'source:source_other_method_signature': ['public StandardToolTipTagFragmentGenerator();'],
 'source_class_fields': [],
 'content:source_class_constructors': ['public StandardToolTipTagFragmentGenerator();'],
 'content:source_class_name': 'StandardToolTipTagFragmentGenerator',
 'con

In [27]:
buggy_source_file_unified = 'data/prompts/buggy_source_data_unified.jsonl'
with open(buggy_source_file_unified, 'w') as f:
    for bug in unified_buggy_bugs:
        f.write(json.dumps(bug) + '\n')

In [28]:
unified_buggy_invoked = [invoked for invoked in unified_buggy_bugs if invoked["focal_method_invoked"]]

In [29]:
buggy_source_file_unified_invoked = 'data/prompts/buggy_source_data_unified_invoked.jsonl'
with open(buggy_source_file_unified_invoked, 'w') as f:
    for bug in unified_buggy_invoked:
        f.write(json.dumps(bug) + '\n')